In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix

# Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# Hide FutureWarning messages for now
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# -----------------------------------------------------------------------------
# 1. Synthetic Biomarker Data Generation (High-dimensional: 100 samples, 1000 features)
# -----------------------------------------------------------------------------
np.random.seed(42)

# Generate a synthetic dataset with 100 patients divided into Control and Disease classes. There are 1000 genes, but only 10 of them are useful for classifying Control vs. Disease
X_raw, y = make_classification(
    n_samples=100,          # n = 100 patients
    n_features=1000,        # p = 1000 genes/biomarkers
    n_informative=10,       # Only 10 genes are true disease drivers
    n_redundant=10,         # Correlated genes
    n_classes=2,            # Control vs. Disease
    random_state=42
)

feature_names = [f"Gene_{i+1}" for i in range(1000)]
df = pd.DataFrame(X_raw, columns=feature_names)

# Split data into training (70%) and test (30%) sets using stratfication to make sure each dataset has similar proportions of Control and Disease patients.
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.3, random_state=42, stratify=y)

# Note that we scale the training set and test set separately to avoid data leakage!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset Loaded: {X_train.shape[0]} train samples, {X_test.shape[0]} test samples, {X_train.shape[1]} biomarkers.\n")


In [ ]:
# -----------------------------------------------------------------------------
# 2. Exploratory Data Analysis
# -----------------------------------------------------------------------------

# Basic dataset information
print("=== Dataset Overview ===")
print(f"Number of samples:       {df.shape[0]}")
print(f"Number of biomarkers:    {df.shape[1]}")
print(f"Training samples:        {X_train.shape[0]}")
print(f"Test samples:            {X_test.shape[0]}")
print(f"Number of disease cases: {sum(y)}")
print(f"Number of controls:      {len(y) - sum(y)}")

# First few rows
print("\n=== First 5 Samples ===")
display(df.head())


In [ ]:
# Summary statistics
print("\n=== Summary Statistics ===")
display(df.describe().round(2))

In [ ]:
# Check for missing values
print("\n=== Missing Values ===")
print(f"Total missing values: {df.isnull().sum().sum()}")

# Class distribution
print("\n=== Class Distribution ===")
print(pd.Series(y).value_counts().rename(index={0: "Control", 1: "Disease"}))

# Check feature variability
print("\n=== Feature Variability ===")
print(f"Mean feature variance: {df.var().mean():.3f}")
print(f"Features with zero variance: {(df.var() == 0).sum()}")

# Check correlation between biomarkers
correlation = df.corr()

print("\n=== Feature Correlation ===")
print(f"Mean absolute correlation: {correlation.abs().values[np.triu_indices_from(correlation, k=1)].mean():.3f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6, 4))

sns.countplot(x=pd.Series(y).map({0: "Control", 1: "Disease"}))

plt.title("Distribution of Patient Groups")
plt.xlabel("Patient Group")
plt.ylabel("Number of Samples")
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

for i in range(5):
    sns.kdeplot(
        df.iloc[:, i],
        label=feature_names[i]
    )

plt.title("Distribution of Selected Biomarkers")
plt.xlabel("Expression Value")
plt.ylabel("Density")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))

sns.heatmap(
    correlation.iloc[:20, :20],
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1
)

plt.title("Correlation Between the First 20 Biomarkers")
plt.show()

In [ ]:
print("=== The High-Dimensional Data Problem ===")

print(f"Number of patients (n):       {df.shape[0]}")
print(f"Number of biomarkers (p):     {df.shape[1]}")
print(f"Biomarkers per patient:       {df.shape[1] / df.shape[0]:.1f}")
print(f"Features per training sample: {X_train.shape[1] / X_train.shape[0]:.1f}")

if df.shape[1] > X_train.shape[0]:
    print("\nWARNING: More biomarkers than training samples!")
    print("This is a high-dimensional dataset (p > n).")

In [ ]:
# -----------------------------------------------------------------------------
# 2. LASSO (L1-Regularized Logistic Regression) for Sparse Feature Selection
# -----------------------------------------------------------------------------

lasso = LogisticRegression(
    l1_ratio=1,
    C=0.1,
    solver='liblinear',
    random_state=42,
    max_iter=1000
)

lasso.fit(X_train_scaled, y_train)

# Predictions
lasso_preds = lasso.predict_proba(X_test_scaled)[:, 1]
lasso_class_preds = lasso.predict(X_test_scaled)

# Model performance
auc = roc_auc_score(y_test, lasso_preds)
accuracy = accuracy_score(y_test, lasso_class_preds)

tn, fp, fn, tp = confusion_matrix(y_test, lasso_class_preds).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

# Selected biomarkers
coefficients = lasso.coef_[0]

lasso_selected_idx = np.where(coefficients != 0)[0]
lasso_selected = np.array(feature_names)[lasso_selected_idx]

# Rank selected biomarkers by absolute coefficient magnitude
selected_results = pd.DataFrame({
    "Biomarker": lasso_selected,
    "Coefficient": coefficients[lasso_selected_idx]
})

selected_results["Abs_Coefficient"] = selected_results["Coefficient"].abs()

selected_results = selected_results.sort_values("Abs_Coefficient", ascending=False)

# Print results
print("=" * 55)
print("                 LASSO RESULTS")
print("=" * 55)

print("\n--- Model Performance ---")
print(f"AUC:          {auc:.3f}")
print(f"Accuracy:     {accuracy:.3f}")
print(f"Sensitivity:  {sensitivity:.3f}")
print(f"Specificity:  {specificity:.3f}")

In [ ]:
print("\n--- Feature Selection ---")
print(f"Original biomarkers:   {len(feature_names)}")
print(f"Selected biomarkers:   {len(lasso_selected)}")
print(f"Eliminated biomarkers: {len(feature_names) - len(lasso_selected)}")
print(f"Reduction:             {(1 - len(lasso_selected)/len(feature_names))*100:.1f}%")

In [ ]:
print("\n--- Selected Biomarkers (ranked by coefficient magnitude) ---")
display(selected_results[["Biomarker", "Coefficient"]].round(4))

In [ ]:
top_features = selected_results.head(15).sort_values("Coefficient")

plt.figure(figsize=(9, 6))

plt.barh(
    top_features["Biomarker"],
    top_features["Coefficient"]
)

plt.axvline(0, color="black", linewidth=0.8)

plt.xlabel("LASSO Coefficient")
plt.ylabel("Biomarker")
plt.title("Top 15 Selected Biomarkers")
plt.tight_layout()
plt.show()

In [ ]:
print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, lasso_class_preds))


In [ ]:
plt.figure(figsize=(5, 4))

sns.heatmap(
    confusion_matrix(y_test, lasso_class_preds),
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Control", "Disease"],
    yticklabels=["Control", "Disease"]
)

plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("LASSO Confusion Matrix")
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# 5A. Support Vector Machine (Linear Kernel)
# -----------------------------------------------------------------------------

from sklearn.svm import SVC
from sklearn.metrics import (roc_auc_score, accuracy_score, confusion_matrix, classification_report)

#svm = CalibratedClassifierCV(SVC(kernel="linear", C=1.0, random_state=42), ensemble=False)
svm = SVC(
    kernel="linear",
    C=1.0,
    probability=True,
    random_state=42
)

svm.fit(X_train_scaled, y_train)

# Predictions
svm_preds = svm.predict_proba(X_test_scaled)[:, 1]
svm_classes = svm.predict(X_test_scaled)

# Performance
svm_auc = roc_auc_score(y_test, svm_preds)
svm_accuracy = accuracy_score(y_test, svm_classes)

tn, fp, fn, tp = confusion_matrix(y_test, svm_classes).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

# Feature weights
svm_weights = pd.Series(svm.coef_[0], index=feature_names)

svm_importance = svm_weights.abs().sort_values(ascending=False)

print("=" * 55)
print("             LINEAR SVM RESULTS")
print("=" * 55)

print("\n--- Model Performance ---")
print(f"AUC:          {svm_auc:.3f}")
print(f"Accuracy:     {svm_accuracy:.3f}")
print(f"Sensitivity:  {sensitivity:.3f}")
print(f"Specificity:  {specificity:.3f}")

print("\n--- Support Vectors ---")
print(f"Total support vectors: {len(svm.support_)}")
print(f"Control support vectors: {svm.n_support_[0]}")
print(f"Disease support vectors: {svm.n_support_[1]}")

print("\n--- Classification Report ---")
print(classification_report(
    y_test, svm_classes,
    target_names=["Control", "Disease"]
))

In [ ]:
print("\n--- Top 15 Biomarkers by Absolute Weight ---")

svm_results = pd.DataFrame({
    "Biomarker": feature_names,
    "Weight": svm.coef_[0],
    "Absolute Weight": np.abs(svm.coef_[0])
}).sort_values("Absolute Weight", ascending=False)

#display(svm_results.head(15).round(4))

# -----------------------------------------------------------------------------
# 5B. Visualize SVM Biomarker Weights
# -----------------------------------------------------------------------------

top_features = svm_results.head(15).sort_values("Weight")

plt.figure(figsize=(9, 6))

plt.barh(
    top_features["Biomarker"],
    top_features["Weight"]
)

plt.axvline(0, color="black", linewidth=0.8)

plt.xlabel("SVM Coefficient")
plt.ylabel("Biomarker")
plt.title("Top 15 Biomarkers: Linear SVM")
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# 5C. SVM ROC Curve
# -----------------------------------------------------------------------------

from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test, svm_preds)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, linewidth=2, label=f"Linear SVM (AUC = {svm_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("SVM Receiver Operating Characteristic")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# 5F. Compare Linear and RBF SVM
# -----------------------------------------------------------------------------

kernels = ["linear", "rbf"]

kernel_results = []

for kernel in kernels:

    model = SVC(
        kernel=kernel,
        C=1.0,
        gamma="scale",
        probability=True,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    predictions = model.predict_proba(X_test_scaled)[:, 1]

    kernel_results.append({
        "Kernel": kernel.upper(),
        "AUC": roc_auc_score(y_test, predictions),
        "Accuracy": model.score(X_test_scaled, y_test)
    })

display(pd.DataFrame(kernel_results).round(3))

In [ ]:
# -----------------------------------------------------------------------------
# 3A. Decision Tree: A Single Tree Making Predictions
# -----------------------------------------------------------------------------

from sklearn.tree import DecisionTreeClassifier, plot_tree

# Train a simple decision tree
tree = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=3,
    random_state=42
)

tree.fit(X_train, y_train)

# Predictions
tree_preds = tree.predict_proba(X_test)[:, 1]
tree_classes = tree.predict(X_test)

print("=" * 50)
print("          SINGLE DECISION TREE")
print("=" * 50)

print(f"AUC:      {roc_auc_score(y_test, tree_preds):.3f}")
print(f"Accuracy: {accuracy_score(y_test, tree_classes):.3f}")

print(f"\nNumber of leaves: {tree.get_n_leaves()}")
print(f"Tree depth:       {tree.get_depth()}")

In [ ]:

# Visualize the decision tree
# Note that blue nodes represent Disease, orange nodes indicate Control, and white nodes indicate ties.
plt.figure(figsize=(20, 10))

plot_tree(
    tree,
    feature_names=feature_names,
    class_names=["Control", "Disease"],
    filled=True,
    rounded=True,
    fontsize=9
)

plt.title("A Single Decision Tree (Maximum Depth = 3)")
plt.show()


In [ ]:
# -----------------------------------------------------------------------------
# 3B. Random Forest: Combining Many Decision Trees
# -----------------------------------------------------------------------------

rf = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Predictions
rf_preds = rf.predict_proba(X_test)[:, 1]
rf_classes = rf.predict(X_test)

# Performance
rf_auc = roc_auc_score(y_test, rf_preds)
rf_accuracy = accuracy_score(y_test, rf_classes)

print("=" * 50)
print("              RANDOM FOREST")
print("=" * 50)

print(f"Number of trees: {rf.n_estimators}")
print(f"AUC:             {rf_auc:.3f}")
print(f"Accuracy:        {rf_accuracy:.3f}")

# Compare against the single decision tree
print("\n--- Model Comparison ---")

comparison = pd.DataFrame({
    "Model": ["Single Decision Tree", "Random Forest"],
    "AUC": [
        roc_auc_score(y_test, tree_preds),
        rf_auc
    ],
    "Accuracy": [
        accuracy_score(y_test, tree_classes),
        rf_accuracy
    ]
})

display(comparison.round(3))

In [ ]:
# -----------------------------------------------------------------------------
# 3C. Random Forest Feature Importance
# -----------------------------------------------------------------------------

rf_importances = pd.Series(
    rf.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

#print("=== Top 15 Biomarkers ===")
#display(rf_importances.head(15).to_frame("Importance").round(4))

# Plot top 15 biomarkers
top_features = rf_importances.head(15).sort_values()

plt.figure(figsize=(9, 6))

plt.barh(
    top_features.index,
    top_features.values
)

plt.xlabel("Feature Importance")
plt.ylabel("Biomarker")
plt.title("Top 15 Biomarkers Identified by Random Forest")
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# Visualize Individual Random Forest Trees
# -----------------------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

for i, ax in enumerate(axes):

    plot_tree(
        rf.estimators_[i],
        feature_names=feature_names,
        class_names=["Control", "Disease"],
        filled=True,
        rounded=True,
        max_depth=2,
        fontsize=7,
        ax=ax
    )

    ax.set_title(f"Random Forest Tree {i+1}")

plt.suptitle("Three Individual Trees from a Random Forest", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# 4. Gradient Boosting
# -----------------------------------------------------------------------------
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
gb.fit(X_train_scaled, y_train)

gb_preds = gb.predict_proba(X_test_scaled)[:, 1]
gb_importances = pd.Series(gb.feature_importances_, index=feature_names).sort_values(ascending=False)
print(f"[Gradient Boosting] AUC: {roc_auc_score(y_test, gb_preds):.3f}")
print(f"[Gradient Boosting] Top 5 Biomarkers:\n{gb_importances.head(5)}\n")

In [ ]:
# -----------------------------------------------------------------------------
# 4A. Compare Three Tree-Based Machine Learning Methods
# -----------------------------------------------------------------------------

models = {
    "Decision Tree": tree,
    "Random Forest": rf,
    "Gradient Boosting": gb
}

comparison = []

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict_proba(X_test)[:, 1]
    classes = model.predict(X_test)

    comparison.append({
        "Model": name,
        "AUC": roc_auc_score(y_test, predictions),
        "Accuracy": accuracy_score(y_test, classes)
    })

model_comparison = pd.DataFrame(comparison)

print("=== Comparison of Tree-Based Models ===")
display(model_comparison.round(3))

In [ ]:
# -----------------------------------------------------------------------------
# 4B. Visualize the First Four Gradient Boosting Trees
# -----------------------------------------------------------------------------

fig, axes = plt.subplots(1, 4, figsize=(24, 6))

for i, ax in enumerate(axes):

    plot_tree(
        gb.estimators_[i, 0],
        feature_names=feature_names,
        filled=True,
        rounded=True,
        max_depth=2,
        fontsize=7,
        ax=ax
    )

    ax.set_title(f"Boosting Tree {i+1}")

plt.suptitle(
    "Gradient Boosting: Sequentially Built Decision Trees",
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# 4C. How Does Gradient Boosting Improve as Trees Are Added?
# -----------------------------------------------------------------------------

stages = [1, 2, 3, 5, 10, 20, 50, 100]

stage_results = []

for n in stages:

    # Obtain predictions after n boosting stages
    predictions = list(gb.staged_predict_proba(X_test))[n-1][:, 1]

    stage_results.append({
        "Number of Trees": n,
        "AUC": roc_auc_score(y_test, predictions)
    })

stage_df = pd.DataFrame(stage_results)

#display(stage_df.round(3))

# Plot performance over boosting stages
plt.figure(figsize=(9, 5))

plt.plot(
    stage_df["Number of Trees"],
    stage_df["AUC"],
    marker="o",
    linewidth=2
)

plt.axhline(
    0.5,
    linestyle="--",
    color="gray",
    label="Random classification"
)

plt.xlabel("Number of Boosting Trees")
plt.ylabel("Test AUC")
plt.title("Gradient Boosting: Performance as Trees Are Added")
plt.xticks(stages)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# 4E. Gradient Boosting Feature Importance
# -----------------------------------------------------------------------------

gb_importances = pd.Series(
    gb.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

print("=== Top 15 Biomarkers Identified by Gradient Boosting ===")

#display(gb_importances.head(15).to_frame("Importance").round(4))

# Visualize the top 15
top_features = gb_importances.head(15).sort_values()

plt.figure(figsize=(9, 6))

plt.barh(
    top_features.index,
    top_features.values
)

plt.xlabel("Feature Importance")
plt.ylabel("Biomarker")
plt.title("Top 15 Gradient Boosting Biomarkers")
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# 6A. Collect Results from All Four Models
# -----------------------------------------------------------------------------

from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
    classification_report, roc_curve
)

models = {
    "LASSO": (lasso_preds, lasso),
    "Random Forest": (rf_preds, rf),
    "Gradient Boosting": (gb_preds, gb),
    "SVM": (svm_preds, svm)
}

model_predictions = {}
performance = []

for name, (probabilities, model) in models.items():

    # Convert probabilities to predicted classes
    predictions = (probabilities >= 0.5).astype(int)

    model_predictions[name] = predictions

    # Calculate confusion matrix
    tn, fp, fn, tp = confusion_matrix(
        y_test, predictions, labels=[0, 1]
    ).ravel()

    # Performance metrics
    performance.append({
        "Model": name,
        "AUC": roc_auc_score(y_test, probabilities),
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Sensitivity": recall_score(y_test, predictions, zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) else 0,
        "F1 Score": f1_score(y_test, predictions, zero_division=0)
    })

performance_df = pd.DataFrame(performance).set_index("Model")

print("=== Overall Model Performance ===")
display(performance_df.round(3))


In [ ]:
# -----------------------------------------------------------------------------
# 6B. Combined ROC Curves
# -----------------------------------------------------------------------------

plt.figure(figsize=(8, 7))

for name, (probabilities, model) in models.items():

    fpr, tpr, thresholds = roc_curve(y_test, probabilities)

    auc = roc_auc_score(y_test, probabilities)

    plt.plot(
        fpr, tpr,
        linewidth=2,
        label=f"{name} (AUC = {auc:.3f})"
    )

plt.plot(
    [0, 1], [0, 1],
    linestyle="--",
    color="gray",
    label="Random classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison: Four Machine Learning Models")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# 6C. Confusion Matrix Comparison
# -----------------------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(11, 9))

for ax, (name, predictions) in zip(axes.flatten(), model_predictions.items()):

    cm = confusion_matrix(y_test, predictions, labels=[0, 1])

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["Control", "Disease"],
        yticklabels=["Control", "Disease"],
        ax=ax
    )

    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrix Comparison", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# 6D. Combined Classification Reports
# -----------------------------------------------------------------------------

reports = []

for name, predictions in model_predictions.items():

    report = classification_report(
        y_test,
        predictions,
        target_names=["Control", "Disease"],
        output_dict=True,
        zero_division=0
    )

    for group in ["Control", "Disease"]:
        reports.append({
            "Model": name,
            "Class": group,
            "Precision": report[group]["precision"],
            "Recall": report[group]["recall"],
            "F1 Score": report[group]["f1-score"]
        })

classification_df = pd.DataFrame(reports)

display(classification_df.round(3))

In [ ]:
# -----------------------------------------------------------------------------
# 6E. Compare Top Biomarkers Identified by Each Algorithm
# -----------------------------------------------------------------------------

feature_scores = {
    "LASSO": pd.Series(
        np.abs(lasso.coef_[0]), index=feature_names
    ),
    "Random Forest": pd.Series(
        rf.feature_importances_, index=feature_names
    ),
    "Gradient Boosting": pd.Series(
        gb.feature_importances_, index=feature_names
    ),
    "SVM": pd.Series(
        np.abs(svm.coef_[0]), index=feature_names
    )
}

top_biomarkers = pd.DataFrame({
    name: scores.sort_values(ascending=False).head(10).index.tolist()
    for name, scores in feature_scores.items()
})

print("=== Top 10 Biomarkers Identified by Each Method ===")
display(top_biomarkers)

In [ ]:
# -----------------------------------------------------------------------------
# 6F. Biomarker Agreement Between Algorithms
# -----------------------------------------------------------------------------

all_selected = pd.concat(
    [top_biomarkers[col] for col in top_biomarkers.columns]
)

agreement = all_selected.value_counts()

print("=== Biomarkers Identified by Multiple Methods ===")
display(
    agreement[agreement >= 2].to_frame("Number of Methods")
)

# Visualize agreement
plt.figure(figsize=(9, 5))

agreement[agreement >= 2].sort_values().plot(
    kind="barh"
)

plt.xlabel("Number of Algorithms Identifying Biomarker")
plt.ylabel("Biomarker")
plt.title("Agreement Between Machine Learning Methods")
plt.xlim(0, 4)
plt.tight_layout()
plt.show()